# BD-KDD Kidney Disease — ANOVA + DODA Feature Selection

## Experimental objective

Evaluate conventional **ANOVA feature selection** against **ANOVA + DODA clinical-priority re-ranking** on the BD-KDD kidney-disease dataset.

### Important design rule

ANOVA scores **all 24 predictors**. DODA also receives **all 24 predictor scores** and performs the final Top-K selection after clinical-priority fusion.

This is essential: if ANOVA selected Top-K before DODA, DODA could not re-prioritize features outside that preliminary Top-K.

### Dataset

- 988 records
- 24 predictors
- Binary target: `Class`
- `Class = 0`: healthy
- `Class = 1`: kidney disease
- 0 missing values
- 80/20 stratified fixed split for the initial experiment
- 5×5 repeated stratified CV for robustness


In [ ]:
# =============================================================================
# STEP 1: LOAD PROCESSED DATASET
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../../data/processed/BD-KDD_cleaned.csv")

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH.resolve()}"
    )

df = pd.read_csv(DATA_PATH)

print("=" * 70)
print("PROCESSED BD-KDD DATASET")
print("=" * 70)
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

display(df.head())


In [ ]:
# =============================================================================
# STEP 2: FEATURE AND TARGET SEPARATION
# =============================================================================

TARGET_COLUMN = "Class"

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print("=" * 70)
print("FEATURE / TARGET SEPARATION")
print("=" * 70)
print(f"Predictors (X) : {X.shape[1]}")
print(f"Records        : {X.shape[0]}")
print(f"Target (y)     : {TARGET_COLUMN}")
print(f"Target values  : {sorted(y.unique().tolist())}")

assert X.shape[1] == 24, "Expected 24 predictors after removing Class."
assert set(y.unique()) == {0, 1}

display(X.head())
display(y.head())


In [ ]:
# =============================================================================
# STEP 3: TARGET DISTRIBUTION
# =============================================================================

target_distribution = pd.DataFrame({
    "Count": y.value_counts().sort_index(),
    "Percentage": (y.value_counts(normalize=True).sort_index() * 100).round(2)
})

display(target_distribution)


In [ ]:
# =============================================================================
# STEP 4: DATA TYPE AND MISSING-VALUE CHECK
# =============================================================================

print("=" * 70)
print("DATA TYPES")
print("=" * 70)
display(X.dtypes.to_frame("Data Type"))

missing = pd.DataFrame({
    "Missing Count": X.isna().sum(),
    "Missing Percentage (%)": (X.isna().mean() * 100).round(2)
})

print("=" * 70)
print("MISSING VALUES")
print("=" * 70)
display(missing)

assert X.isna().sum().sum() == 0, "Unexpected missing values detected."

print("No missing values detected. No imputation will be performed.")


## Preprocessing decision

The EDA established that BD-KDD contains **zero missing values**. Therefore, no imputation step is included in this notebook.

Feature selection is fitted only on training data/folds. Model scaling is applied only where needed and is fitted only on training data/folds.


In [ ]:
# =============================================================================
# STEP 5: FIXED 80/20 STRATIFIED TRAIN-TEST SPLIT
# =============================================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("=" * 70)
print("TRAIN-TEST SPLIT")
print("=" * 70)
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

print("\nTraining class distribution:")
display((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTesting class distribution:")
display((y_test.value_counts(normalize=True) * 100).round(2))


In [ ]:
# =============================================================================
# STEP 6: FEATURE-SCALE CHECK
# =============================================================================
# ANOVA F-statistics are scale-invariant, so feature selection is performed
# on the original training-fold values.
#
# Scaling is applied later only for Logistic Regression.

print("ANOVA will be fitted on the original training data.")
print("Logistic Regression will be scaled after feature selection.")
print("Random Forest and XGBoost will use the selected features without scaling.")


# 1. ANOVA Baseline

ANOVA is used as the statistical feature-selection baseline.

**Important:** ANOVA scores all 24 predictors first. Top-K is applied only after the complete ranking has been generated.


In [ ]:
# =============================================================================
# STEP 7: ANOVA SCORES FOR ALL FEATURES
# =============================================================================

from sklearn.feature_selection import SelectKBest, f_classif

anova_selector = SelectKBest(
    score_func=f_classif,
    k="all"
)

anova_selector.fit(X_train, y_train)

anova_scores = pd.DataFrame({
    "Feature": X_train.columns,
    "ANOVA_F_Score": anova_selector.scores_,
    "ANOVA_P_Value": anova_selector.pvalues_
}).sort_values(
    "ANOVA_F_Score",
    ascending=False
).reset_index(drop=True)

anova_scores["ANOVA_Rank"] = np.arange(1, len(anova_scores) + 1)

display(anova_scores)


In [ ]:
# =============================================================================
# STEP 8: TOP-K ANOVA FEATURE SETS
# =============================================================================

TOP_K_VALUES = [5, 10, 15, 20]

print(f"Total predictors: {X.shape[1]}")
print(f"Top-K values: {TOP_K_VALUES}")

anova_results = {}

for top_k in TOP_K_VALUES:

    selected = anova_scores.head(top_k)["Feature"].tolist()

    anova_results[top_k] = {
        "features": selected,
        "X_train": X_train[selected].copy(),
        "X_test": X_test[selected].copy()
    }

    print("=" * 70)
    print(f"ANOVA TOP-{top_k}")
    print("=" * 70)
    print(selected)


In [ ]:
# =============================================================================
# STEP 9: DOWNSTREAM MODELS
# =============================================================================

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}

print(list(models.keys()))


In [ ]:
# =============================================================================
# STEP 10: ANOVA BASELINE MODEL EVALUATION
# =============================================================================

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

fixed_anova_rows = []

for top_k in TOP_K_VALUES:

    Xtr_selected = anova_results[top_k]["X_train"]
    Xte_selected = anova_results[top_k]["X_test"]
    selected_features = anova_results[top_k]["features"]

    for model_name, model_template in models.items():

        model = model_template.__class__(
            **model_template.get_params()
        )

        Xtr = Xtr_selected
        Xte = Xte_selected

        if model_name == "Logistic Regression":
            scaler = StandardScaler()
            Xtr = scaler.fit_transform(Xtr)
            Xte = scaler.transform(Xte)

        model.fit(Xtr, y_train)

        y_pred = model.predict(Xte)
        y_prob = model.predict_proba(Xte)[:, 1]

        fixed_anova_rows.append({
            "Method": "ANOVA",
            "Top_K": top_k,
            "Model": model_name,
            "Selected_Features": ", ".join(selected_features),
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1": f1_score(y_test, y_pred, zero_division=0),
            "ROC_AUC": roc_auc_score(y_test, y_prob)
        })

fixed_anova_results_df = pd.DataFrame(fixed_anova_rows)
display(fixed_anova_results_df.round(4))


In [ ]:
# =============================================================================
# STEP 11: SAVE ANOVA BASELINE
# =============================================================================

baseline_dir = Path("../../../results/kidney_disease/baseline")
baseline_dir.mkdir(parents=True, exist_ok=True)

baseline_path = baseline_dir / "anova_baseline_results.csv"
fixed_anova_results_df.to_csv(baseline_path, index=False)

print(f"Saved: {baseline_path.resolve()}")


# 2. DODA Clinical Re-ranking

### Correct experimental order

```text
24 original predictors
        ↓
ANOVA scores all 24
        ↓
statistical ranking
        +
pre-specified clinical weights
        ↓
DODA rank fusion
        ↓
final Top-K
```

DODA must **not** receive an already-truncated ANOVA Top-K set. Otherwise, clinically prioritized features outside that preliminary Top-K would be impossible to recover.


In [ ]:
# =============================================================================
# STEP 12: INSTALL / IMPORT DODA
# =============================================================================

# Run the following installation once if DODA is not installed:
# %pip install --no-cache-dir git+https://github.com/anandha-3679/DODA.git

import doda

from doda import DODASelector
from doda.adapters import SklearnAdapter
from doda.knowledge.providers import JSONProvider
from doda.fusion import RankFusion

print("DODA:", doda.__file__)

print("Provider:", JSONProvider)
print("Fusion:", RankFusion)

### Why a custom provider is needed here

`BD_KDD_clinical_weights.json` uses a richer schema than the package's `JSONProvider` expects — the per-feature weights live under a nested `"weights"` key, alongside `schema_version`, `sources`, `feature_tiers`, and other provenance metadata. `JSONProvider.get_weights()` only ever looks at the **top level** of the loaded JSON, so for every feature it does `self.weights.get("Age", 1.0)` — and since `"Age"` is not a top-level key in this file, **every single feature silently fell back to the default weight of 1.0**. This made the clinical layer completely flat (a tie across all 24 features), which is exactly why DODA wasn't producing any rank changes: Rank Fusion had a real statistical ranking to combine with, but a clinical ranking with no information in it at all.

This was previously masked by the validation cell below, which manually did `weights_payload["weights"]` to check coverage — that check is correct, but it validates a *different* object than the one actually passed into `DODASelector`, so it passed even though the real `provider` was broken.

**Fix**: `NestedJSONProvider`, defined below, implements the same `BaseKnowledgeProvider` interface as `JSONProvider` but reads from the nested `"weights"` key. The clinical weights file itself does not need to change — it's correctly structured for the richer schema; the loading code needed to catch up to it. If this schema becomes the standard going forward, `JSONProvider` in the actual `doda` package should be updated to check for a `"weights"` key and fall back to top-level parsing for older flat files, so every notebook doesn't need its own local copy of this class.

In [ ]:
# =============================================================================
# STEP 12B: NESTED-SCHEMA CLINICAL KNOWLEDGE PROVIDER
# =============================================================================

from doda.knowledge.providers.base import BaseKnowledgeProvider
import json as _json


class NestedJSONProvider(BaseKnowledgeProvider):
    """
    Reads clinical weights from the "weights" key of a richer JSON schema
    (schema_version, target, tier_mapping, feature_tiers, sources, etc.),
    rather than expecting the weights at the top level the way the
    package's built-in JSONProvider does.

    Same fallback behavior as JSONProvider otherwise: a feature not present
    in the weights dict gets a neutral default of 1.0, not an error --
    kept intentionally consistent so debugging behavior doesn't change,
    only where the weights are actually read from.
    """

    def __init__(self, path):
        self.path = path
        with open(path, "r", encoding="utf-8") as f:
            payload = _json.load(f)

        if "weights" not in payload:
            raise KeyError(
                f"{path} has no top-level 'weights' key -- "
                "this loader expects the nested schema. Use JSONProvider "
                "instead for a flat {feature: weight} file."
            )

        self.weights = payload["weights"]

    def get_weights(self, feature_names):
        return {
            feature: self.weights.get(feature, 1.0)
            for feature in feature_names
        }


print("NestedJSONProvider defined.")

In [ ]:
# =============================================================================
# STEP 13: LOCATE AND VALIDATE CLINICAL WEIGHTS
# =============================================================================

WEIGHTS_FILENAME = "BD_KDD_clinical_weights.json"

search_roots = [Path.cwd(), *Path.cwd().parents]
matches = []

for root in search_roots:
    try:
        matches.extend(root.rglob(WEIGHTS_FILENAME))
    except (PermissionError, OSError):
        continue

# Remove duplicates while preserving order
matches = list(dict.fromkeys(matches))

if not matches:
    raise FileNotFoundError(
        f"{WEIGHTS_FILENAME} was not found. "
        "Place it in the repository and rerun this cell."
    )

WEIGHTS_PATH = matches[0]

print(f"Using clinical weights: {WEIGHTS_PATH.resolve()}")

provider = NestedJSONProvider(str(WEIGHTS_PATH))

# Validate coverage using the JSON file directly.
import json

with open(WEIGHTS_PATH, "r", encoding="utf-8") as f:
    weights_payload = json.load(f)

clinical_weights = weights_payload["weights"]

missing_weights = sorted(set(X.columns) - set(clinical_weights))
extra_weights = sorted(set(clinical_weights) - set(X.columns))

print(f"Clinical weights: {len(clinical_weights)}")
print(f"Predictors:       {len(X.columns)}")
print(f"Missing weights:  {missing_weights}")
print(f"Extra weights:    {extra_weights}")

assert not missing_weights
assert not extra_weights


In [ ]:
# =============================================================================
# STEP 14: DODA — SCORE ALL FEATURES, THEN APPLY TOP-K
# =============================================================================

rank_doda_results = {}

for top_k in TOP_K_VALUES:

    anova_operator = SklearnAdapter(
        SelectKBest(
            score_func=f_classif,
            k="all"     # CRITICAL: all 24 features reach DODA
        )
    )

    doda_selector = DODASelector(
        operators=[anova_operator],
        provider=provider,
        fusion=RankFusion(),
        top_k=top_k
    )

    # Fit only on training data.
    doda_selector.fit(
        X_train,
        y_train
    )

    selected_features = list(
        doda_selector.get_selected_features()
    )

    rank_doda_results[top_k] = {
        "selector": doda_selector,
        "features": selected_features,
        "X_train": X_train[selected_features].copy(),
        "X_test": X_test[selected_features].copy()
    }

    print("=" * 70)
    print(f"DODA TOP-{top_k}")
    print("=" * 70)
    print(selected_features)
    print(f"Number selected: {len(selected_features)}")

    assert len(selected_features) == top_k


In [ ]:
# =============================================================================
# STEP 15: INSPECT DODA RANK CHANGES
# =============================================================================

INSPECTION_TOP_K = 10

selector = rank_doda_results[INSPECTION_TOP_K]["selector"]

print("=" * 70)
print(f"DODA RANK COMPARISON — TOP-{INSPECTION_TOP_K}")
print("=" * 70)

rank_comparison = selector.compare_scores()

display(rank_comparison)

print("\nSelected features:")
print(rank_doda_results[INSPECTION_TOP_K]["features"])


In [ ]:
# =============================================================================
# STEP 15B: EXPLICIT ANOVA vs DODA RANK-SHIFT TABLE
# =============================================================================
# This table separates:
#   - statistical ANOVA rank
#   - DODA final rank
#   - clinical weight
#   - rank movement
#
# Positive Rank_Change means the feature moved upward.
# Example: ANOVA rank 10 -> DODA rank 4 = +6
#
# IMPORTANT:
# The selector is explicitly retrieved from rank_doda_results
# for the selected Top-K value. This avoids relying on a
# previously defined 'selector' variable.

# -------------------------------------------------------------------------
# Select which DODA Top-K experiment to inspect
# -------------------------------------------------------------------------

k = 10

if k not in rank_doda_results:
    raise KeyError(
        f"Top-K={k} was not found in rank_doda_results. "
        f"Available values: {list(rank_doda_results.keys())}"
    )

selector = rank_doda_results[k]["selector"]


# -------------------------------------------------------------------------
# ANOVA / mathematical ranking
# -------------------------------------------------------------------------

math_rank = pd.DataFrame(
    selector.math_scores_.items(),
    columns=[
        "Feature",
        "ANOVA_Normalized_Score"
    ]
)

math_rank["ANOVA_Rank"] = (
    math_rank["ANOVA_Normalized_Score"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


# -------------------------------------------------------------------------
# DODA final ranking
# -------------------------------------------------------------------------

final_rank = pd.DataFrame(
    selector.final_scores_.items(),
    columns=[
        "Feature",
        "DODA_Final_Score"
    ]
)

final_rank["DODA_Rank"] = (
    final_rank["DODA_Final_Score"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


# -------------------------------------------------------------------------
# Combine rankings
# -------------------------------------------------------------------------

rank_shift_df = (
    math_rank[
        [
            "Feature",
            "ANOVA_Normalized_Score",
            "ANOVA_Rank"
        ]
    ]
    .merge(
        final_rank[
            [
                "Feature",
                "DODA_Final_Score",
                "DODA_Rank"
            ]
        ],
        on="Feature",
        how="inner"
    )
)


# -------------------------------------------------------------------------
# Add clinical weights
# -------------------------------------------------------------------------

rank_shift_df["Clinical_Weight"] = (
    rank_shift_df["Feature"]
    .map(selector.clinical_weights_)
)


# -------------------------------------------------------------------------
# Calculate rank movement
# -------------------------------------------------------------------------
# Positive value  = moved upward
# Negative value  = moved downward
# Zero             = no rank movement

rank_shift_df["Rank_Change"] = (
    rank_shift_df["ANOVA_Rank"]
    - rank_shift_df["DODA_Rank"]
)


# -------------------------------------------------------------------------
# Sort by DODA final rank
# -------------------------------------------------------------------------

rank_shift_df = (
    rank_shift_df
    .sort_values(
        by="DODA_Rank"
    )
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------
# Display
# -------------------------------------------------------------------------

print("=" * 80)
print(f"ANOVA vs DODA RANK-SHIFT ANALYSIS — TOP-{k}")
print("=" * 80)

display(rank_shift_df)

In [ ]:
# =============================================================================
# STEP 16: INSPECT CLINICAL WEIGHTS
# =============================================================================

clinical_weight_table = pd.DataFrame({
    "Feature": list(selector.clinical_weights_.keys()),
    "Clinical_Weight": list(selector.clinical_weights_.values())
}).sort_values(
    "Clinical_Weight",
    ascending=False
).reset_index(drop=True)

display(clinical_weight_table)


In [ ]:
# =============================================================================
# STEP 17: DODA MODEL EVALUATION — FIXED SPLIT
# =============================================================================

fixed_doda_rows = []

for top_k in TOP_K_VALUES:

    Xtr_selected = rank_doda_results[top_k]["X_train"]
    Xte_selected = rank_doda_results[top_k]["X_test"]
    selected_features = rank_doda_results[top_k]["features"]

    for model_name, model_template in models.items():

        model = model_template.__class__(
            **model_template.get_params()
        )

        Xtr = Xtr_selected
        Xte = Xte_selected

        if model_name == "Logistic Regression":
            scaler = StandardScaler()
            Xtr = scaler.fit_transform(Xtr)
            Xte = scaler.transform(Xte)

        model.fit(Xtr, y_train)

        y_pred = model.predict(Xte)
        y_prob = model.predict_proba(Xte)[:, 1]

        fixed_doda_rows.append({
            "Method": "DODA",
            "Top_K": top_k,
            "Model": model_name,
            "Selected_Features": ", ".join(selected_features),
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1": f1_score(y_test, y_pred, zero_division=0),
            "ROC_AUC": roc_auc_score(y_test, y_prob)
        })

fixed_doda_results_df = pd.DataFrame(fixed_doda_rows)
display(fixed_doda_results_df.round(4))


In [ ]:
# =============================================================================
# STEP 18: FIXED-SPLIT ANOVA vs DODA COMPARISON
# =============================================================================

fixed_comparison_df = pd.concat(
    [fixed_anova_results_df, fixed_doda_results_df],
    ignore_index=True
)

display(fixed_comparison_df.round(4))


In [ ]:
# =============================================================================
# STEP 19: COMPARE FEATURE SETS ON THE FIXED SPLIT
# =============================================================================

fixed_feature_comparison = []

for top_k in TOP_K_VALUES:

    anova_set = set(anova_results[top_k]["features"])
    doda_set = set(rank_doda_results[top_k]["features"])

    intersection = anova_set & doda_set
    union = anova_set | doda_set

    jaccard = len(intersection) / len(union)

    fixed_feature_comparison.append({
        "Top_K": top_k,
        "ANOVA_Features": ", ".join(anova_results[top_k]["features"]),
        "DODA_Features": ", ".join(rank_doda_results[top_k]["features"]),
        "Same_Feature_Set": anova_set == doda_set,
        "Overlap_Count": len(intersection),
        "Jaccard": jaccard,
        "Features_Changed": top_k - len(intersection)
    })

fixed_feature_comparison_df = pd.DataFrame(fixed_feature_comparison)

display(fixed_feature_comparison_df)


In [ ]:
# =============================================================================
# STEP 20: SAVE FIXED-SPLIT DODA RESULTS
# =============================================================================

doda_dir = Path("../../../results/kidney_disease/doda")
doda_dir.mkdir(parents=True, exist_ok=True)

fixed_doda_results_df.to_csv(
    doda_dir / "anova_doda_fixed_split_results.csv",
    index=False
)

fixed_feature_comparison_df.to_csv(
    doda_dir / "anova_doda_fixed_split_feature_comparison.csv",
    index=False
)

print(f"Saved results to: {doda_dir.resolve()}")


# 3.1 Five-Fold Stratified Cross-Validation

The fixed 80/20 split is followed by a separate **5-fold stratified cross-validation** experiment.

Feature selection is refit inside each training fold. DODA also receives all features and performs the final Top-K selection inside the training fold. Logistic Regression scaling is fitted only on the corresponding training fold.

The DODA fusion method remains **RankFusion**, matching the current BD-KDD pipeline.

In [ ]:
# =============================================================================
# STEP 21: 5-FOLD CV CONFIGURATION
# =============================================================================
from sklearn.model_selection import StratifiedKFold

CV5_SPLITS = 5
CV5_RANDOM_STATE = 42
cv5 = StratifiedKFold(n_splits=CV5_SPLITS, shuffle=True, random_state=CV5_RANDOM_STATE)
cv5_results = []

print("5-fold CV configuration loaded.")
print("Splits:", CV5_SPLITS)
print("Top-K values:", TOP_K_VALUES)
print("Fusion: RankFusion")


In [ ]:
# =============================================================================
# STEP 22: 5-FOLD CV — ANOVA vs DODA
# =============================================================================

for fold, (train_idx, val_idx) in enumerate(cv5.split(X, y), start=1):
    X_train_fold = X.iloc[train_idx].copy()
    X_val_fold = X.iloc[val_idx].copy()
    y_train_fold = y.iloc[train_idx].copy()
    y_val_fold = y.iloc[val_idx].copy()

    anova_fold = SelectKBest(score_func=f_classif, k="all")
    anova_fold.fit(X_train_fold, y_train_fold)
    anova_fold_scores = pd.DataFrame({
        "Feature": X_train_fold.columns,
        "Score": anova_fold.scores_
    }).sort_values("Score", ascending=False)

    for top_k in TOP_K_VALUES:
        anova_features = anova_fold_scores.head(top_k)["Feature"].tolist()

        fold_doda = DODASelector(
            operators=[SklearnAdapter(SelectKBest(score_func=f_classif, k="all"))],
            provider=NestedJSONProvider(str(WEIGHTS_PATH)),
            fusion=RankFusion(),
            top_k=top_k
        )
        fold_doda.fit(X_train_fold, y_train_fold)
        doda_features = list(fold_doda.get_selected_features())

        feature_sets = {"ANOVA": anova_features, "DODA": doda_features}

        for method, selected_features in feature_sets.items():
            Xtr_selected = X_train_fold[selected_features]
            Xval_selected = X_val_fold[selected_features]

            for model_name, model_template in models.items():
                model = model_template.__class__(**model_template.get_params())
                Xtr, Xval = Xtr_selected, Xval_selected

                if model_name == "Logistic Regression":
                    scaler = StandardScaler()
                    Xtr = scaler.fit_transform(Xtr)
                    Xval = scaler.transform(Xval)

                model.fit(Xtr, y_train_fold)
                y_pred = model.predict(Xval)
                y_prob = model.predict_proba(Xval)[:, 1]

                cv5_results.append({
                    "Fold": fold,
                    "Method": method,
                    "Top_K": top_k,
                    "Model": model_name,
                    "Accuracy": accuracy_score(y_val_fold, y_pred),
                    "Precision": precision_score(y_val_fold, y_pred, zero_division=0),
                    "Recall": recall_score(y_val_fold, y_pred, zero_division=0),
                    "F1": f1_score(y_val_fold, y_pred, zero_division=0),
                    "ROC_AUC": roc_auc_score(y_val_fold, y_prob),
                    "Selected_Features": ", ".join(selected_features)
                })

cv5_results_df = pd.DataFrame(cv5_results)
assert len(cv5_results_df) == 5 * len(TOP_K_VALUES) * 2 * len(models)
print("Completed 5-fold CV. Rows:", len(cv5_results_df))
display(cv5_results_df.head())


In [ ]:
# =============================================================================
# STEP 23: 5-FOLD CV PERFORMANCE SUMMARY
# =============================================================================
cv5_summary_df = (
    cv5_results_df.groupby(["Method", "Top_K", "Model"])
    [["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]]
    .agg(["mean", "std"])
    .reset_index()
)
cv5_summary_df.columns = [
    "Method", "Top_K", "Model", "Accuracy_Mean", "Accuracy_STD",
    "Precision_Mean", "Precision_STD", "Recall_Mean", "Recall_STD",
    "F1_Mean", "F1_STD", "ROC_AUC_Mean", "ROC_AUC_STD"
]
display(cv5_summary_df.round(4))

cv5_dir = Path("../../../results/kidney_disease/cv")
cv5_dir.mkdir(parents=True, exist_ok=True)
cv5_results_df.to_csv(cv5_dir / "anova_doda_5fold_results.csv", index=False)
cv5_summary_df.to_csv(cv5_dir / "anova_doda_5fold_summary.csv", index=False)


# 3.2 Robustness: 5×5 Repeated Stratified Cross-Validation

The fixed split provides a reproducible initial comparison.

For robustness, feature selection is **recomputed inside every training fold**. This prevents information from the validation fold from influencing ANOVA or DODA.

There are:

- 5 folds
- 5 repeats
- 25 train/validation runs
- 4 Top-K budgets
- 2 feature-selection methods
- 3 downstream models

This produces 600 model-evaluation rows.


In [ ]:
# =============================================================================
# STEP 24: REPEATED CV CONFIGURATION
# =============================================================================

from sklearn.model_selection import RepeatedStratifiedKFold

TOP_K_VALUES = [5, 10, 15, 20]

N_SPLITS = 5
N_REPEATS = 5
RANDOM_STATE = 42

cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE
)

print(f"Top-K values: {TOP_K_VALUES}")
print(f"CV runs: {N_SPLITS * N_REPEATS}")


In [ ]:
# =============================================================================
# STEP 25: 5×5 REPEATED CV — ANOVA vs DODA
# =============================================================================

cv_results = []

for run_id, (train_idx, val_idx) in enumerate(
    cv.split(X, y),
    start=1
):

    X_train_fold = X.iloc[train_idx].copy()
    X_val_fold = X.iloc[val_idx].copy()

    y_train_fold = y.iloc[train_idx].copy()
    y_val_fold = y.iloc[val_idx].copy()

    print(f"Run {run_id:02d}/{N_SPLITS * N_REPEATS}")

    # -----------------------------------------------------------------
    # ANOVA scores ALL features inside this training fold
    # -----------------------------------------------------------------

    anova_all = SelectKBest(
        score_func=f_classif,
        k="all"
    )

    anova_all.fit(
        X_train_fold,
        y_train_fold
    )

    anova_fold_scores = pd.DataFrame({
        "Feature": X_train_fold.columns,
        "Score": anova_all.scores_
    }).sort_values(
        "Score",
        ascending=False
    )

    # -----------------------------------------------------------------
    # DODA also receives ALL features and performs final Top-K
    # -----------------------------------------------------------------

    # NOTE: previously this cell also fit a `doda_selector` once here at
    # top_k=max(TOP_K_VALUES) before the loop below -- that result was
    # never used (the loop always builds its own fold_doda per top_k),
    # so the redundant fit was removed to cut this cell's runtime.

    for top_k in TOP_K_VALUES:

        # -------------------------
        # ANOVA Top-K
        # -------------------------

        anova_features = (
            anova_fold_scores
            .head(top_k)["Feature"]
            .tolist()
        )

        # -------------------------
        # DODA Top-K
        # -------------------------

        fold_doda_operator = SklearnAdapter(
            SelectKBest(
                score_func=f_classif,
                k="all"
            )
        )

        fold_doda = DODASelector(
            operators=[fold_doda_operator],
            provider=NestedJSONProvider(str(WEIGHTS_PATH)),
            fusion=RankFusion(),
            top_k=top_k
        )

        fold_doda.fit(
            X_train_fold,
            y_train_fold
        )

        doda_features = list(
            fold_doda.get_selected_features()
        )

        # -------------------------
        # Model loop
        # -------------------------

        feature_sets = {
            "ANOVA": anova_features,
            "DODA": doda_features
        }

        for method, selected_features in feature_sets.items():

            Xtr_selected = X_train_fold[selected_features]
            Xval_selected = X_val_fold[selected_features]

            for model_name, model_template in models.items():

                model = model_template.__class__(
                    **model_template.get_params()
                )

                Xtr = Xtr_selected
                Xval = Xval_selected

                if model_name == "Logistic Regression":
                    scaler = StandardScaler()
                    Xtr = scaler.fit_transform(Xtr)
                    Xval = scaler.transform(Xval)

                model.fit(Xtr, y_train_fold)

                y_pred = model.predict(Xval)
                y_prob = model.predict_proba(Xval)[:, 1]

                cv_results.append({
                    "Run": run_id,
                    "Method": method,
                    "Top_K": top_k,
                    "Model": model_name,
                    "Accuracy": accuracy_score(y_val_fold, y_pred),
                    "Precision": precision_score(
                        y_val_fold, y_pred, zero_division=0
                    ),
                    "Recall": recall_score(
                        y_val_fold, y_pred, zero_division=0
                    ),
                    "F1": f1_score(
                        y_val_fold, y_pred, zero_division=0
                    ),
                    "ROC_AUC": roc_auc_score(
                        y_val_fold, y_prob
                    ),
                    "Selected_Features": ", ".join(selected_features)
                })

cv_results_df = pd.DataFrame(cv_results)

print("Completed.")
print(f"Rows generated: {len(cv_results_df)}")

assert len(cv_results_df) == 25 * 4 * 2 * 3

display(cv_results_df.head())


In [ ]:
# =============================================================================
# STEP 26: CV PERFORMANCE SUMMARY
# =============================================================================

cv_summary_df = (
    cv_results_df
    .groupby(["Method", "Top_K", "Model"])
    [["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]]
    .agg(["mean", "std"])
    .reset_index()
)

cv_summary_df.columns = [
    "Method", "Top_K", "Model",
    "Accuracy_Mean", "Accuracy_STD",
    "Precision_Mean", "Precision_STD",
    "Recall_Mean", "Recall_STD",
    "F1_Mean", "F1_STD",
    "ROC_AUC_Mean", "ROC_AUC_STD"
]

display(cv_summary_df.round(4))


In [ ]:
# =============================================================================
# STEP 24: SAVE REPEATED-CV RESULTS
# =============================================================================

cv_dir = Path("../../../results/kidney_disease/cv")
cv_dir.mkdir(parents=True, exist_ok=True)

cv_results_df.to_csv(
    cv_dir / "anova_vs_doda_5x5_repeated_cv_results.csv",
    index=False
)

cv_summary_df.to_csv(
    cv_dir / "anova_vs_doda_5x5_repeated_cv_summary.csv",
    index=False
)

print(f"Saved to: {cv_dir.resolve()}")


# 4. Feature-Selection Stability, Rank/Set Change, and Statistical Comparison

Two different concepts are reported:

- **Feature-set change:** whether the Top-K membership changes.
- **Rank change:** whether feature ordering changes even when membership is retained.

Jaccard similarity evaluates set overlap, so it does not measure ordering. The fixed-split DODA rank comparison is used to inspect ordering changes.


In [ ]:
# =============================================================================
# STEP 28: FEATURE-SET COMPARISON ACROSS THE 25 CV RUNS
# =============================================================================

feature_comparison_rows = []

for run_id in range(1, 26):

    for top_k in TOP_K_VALUES:

        anova_row = cv_results_df[
            (cv_results_df["Run"] == run_id) &
            (cv_results_df["Method"] == "ANOVA") &
            (cv_results_df["Top_K"] == top_k)
        ].iloc[0]

        doda_row = cv_results_df[
            (cv_results_df["Run"] == run_id) &
            (cv_results_df["Method"] == "DODA") &
            (cv_results_df["Top_K"] == top_k)
        ].iloc[0]

        anova_set = set(
            anova_row["Selected_Features"].split(", ")
        )
        doda_set = set(
            doda_row["Selected_Features"].split(", ")
        )

        overlap = len(anova_set & doda_set)
        union = len(anova_set | doda_set)

        feature_comparison_rows.append({
            "Run": run_id,
            "Top_K": top_k,
            "Same_Feature_Set": anova_set == doda_set,
            "Overlap_Count": overlap,
            "Jaccard": overlap / union,
            "Features_Changed": top_k - overlap
        })

feature_comparison_df = pd.DataFrame(
    feature_comparison_rows
)

display(
    feature_comparison_df
    .groupby("Top_K")
    [["Same_Feature_Set", "Overlap_Count", "Jaccard", "Features_Changed"]]
    .agg(["mean", "std"])
    .round(4)
)


In [ ]:
# =============================================================================
# STEP 29: FEATURE-SELECTION STABILITY
# =============================================================================

from itertools import combinations

stability_rows = []

for method in ["ANOVA", "DODA"]:

    for top_k in TOP_K_VALUES:

        subset = cv_results_df[
            (cv_results_df["Method"] == method) &
            (cv_results_df["Top_K"] == top_k)
        ]

        # One feature set per run; model does not affect selection.
        run_sets = {}

        for run_id in range(1, 26):
            row = subset[
                subset["Run"] == run_id
            ].iloc[0]

            run_sets[run_id] = set(
                row["Selected_Features"].split(", ")
            )

        pairwise_jaccards = []

        for a, b in combinations(range(1, 26), 2):

            set_a = run_sets[a]
            set_b = run_sets[b]

            pairwise_jaccards.append(
                len(set_a & set_b) / len(set_a | set_b)
            )

        stability_rows.append({
            "Method": method,
            "Top_K": top_k,
            "Mean_Jaccard": np.mean(pairwise_jaccards),
            "SD_Jaccard": np.std(pairwise_jaccards, ddof=1),
            "Min_Jaccard": np.min(pairwise_jaccards),
            "Max_Jaccard": np.max(pairwise_jaccards)
        })

stability_df = pd.DataFrame(stability_rows)

display(stability_df.round(4))


## 4.1 Additional Feature-Selection Stability Analyses

The 25 repeated-CV runs are used to quantify selection frequency, chance-adjusted stability, and direct ANOVA–DODA feature-set agreement.

In [ ]:
# =============================================================================
# STEP 30: FEATURE SELECTION FREQUENCY
# =============================================================================
frequency_rows = []
for method in ["ANOVA", "DODA"]:
    for top_k in TOP_K_VALUES:
        subset = cv_results_df[(cv_results_df["Method"] == method) & (cv_results_df["Top_K"] == top_k)]
        counts = {feature: 0 for feature in X.columns}
        for run_id in sorted(subset["Run"].unique()):
            row = subset[subset["Run"] == run_id].iloc[0]
            for feature in row["Selected_Features"].split(", "):
                counts[feature] += 1
        for feature, count in counts.items():
            frequency_rows.append({
                "Method": method, "Top_K": top_k, "Feature": feature,
                "Selection_Count": count,
                "Selection_Frequency": count / (N_SPLITS * N_REPEATS)
            })
feature_frequency_df = pd.DataFrame(frequency_rows)

display(
    feature_frequency_df.sort_values(
        ["Method", "Top_K", "Selection_Frequency", "Feature"],
        ascending=[True, True, False, True]
    ).groupby(["Method", "Top_K"], group_keys=False).head(10).round(4)
)


In [ ]:
# =============================================================================
# STEP 31: KUNCHEVA INDEX
# =============================================================================
from itertools import combinations
N_FEATURES = X.shape[1]
kuncheva_rows = []

for method in ["ANOVA", "DODA"]:
    for top_k in TOP_K_VALUES:
        subset = cv_results_df[(cv_results_df["Method"] == method) & (cv_results_df["Top_K"] == top_k)]
        run_sets = {}
        for run_id in sorted(subset["Run"].unique()):
            row = subset[subset["Run"] == run_id].iloc[0]
            run_sets[run_id] = set(row["Selected_Features"].split(", "))

        for run_a, run_b in combinations(sorted(run_sets), 2):
            intersection = len(run_sets[run_a] & run_sets[run_b])
            denominator = top_k * (N_FEATURES - top_k)
            value = np.nan if denominator == 0 else ((intersection * N_FEATURES) - (top_k ** 2)) / denominator
            kuncheva_rows.append({
                "Method": method, "Top_K": top_k, "Run_A": run_a, "Run_B": run_b,
                "Intersection": intersection, "Kuncheva_Index": value
            })

kuncheva_df = pd.DataFrame(kuncheva_rows)
kuncheva_summary_df = (
    kuncheva_df.groupby(["Method", "Top_K"])["Kuncheva_Index"]
    .agg(Mean_Kuncheva="mean", Std_Kuncheva="std", Min_Kuncheva="min", Max_Kuncheva="max")
    .reset_index()
)
display(kuncheva_summary_df.round(4))


In [ ]:
# =============================================================================
# STEP 32: ANOVA–DODA FEATURE-SET AGREEMENT SUMMARY
# =============================================================================
agreement_summary_df = (
    feature_comparison_df.groupby("Top_K")
    .agg(
        Same_Feature_Set_Rate=("Same_Feature_Set", "mean"),
        Mean_Overlap=("Overlap_Count", "mean"),
        SD_Overlap=("Overlap_Count", "std"),
        Mean_Jaccard=("Jaccard", "mean"),
        SD_Jaccard=("Jaccard", "std"),
        Mean_Features_Changed=("Features_Changed", "mean"),
        SD_Features_Changed=("Features_Changed", "std")
    ).reset_index()
)
display(agreement_summary_df.round(4))


In [ ]:
# =============================================================================
# STEP 33: PAIRED STATISTICAL COMPARISON OF PREDICTIVE PERFORMANCE
# =============================================================================

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

metrics = ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]

comparison_rows = []

for top_k in TOP_K_VALUES:

    for model_name in models.keys():

        anova = cv_results_df[
            (cv_results_df["Method"] == "ANOVA") &
            (cv_results_df["Top_K"] == top_k) &
            (cv_results_df["Model"] == model_name)
        ].sort_values("Run")

        doda = cv_results_df[
            (cv_results_df["Method"] == "DODA") &
            (cv_results_df["Top_K"] == top_k) &
            (cv_results_df["Model"] == model_name)
        ].sort_values("Run")

        assert len(anova) == 25
        assert len(doda) == 25
        assert (anova["Run"].values == doda["Run"].values).all()

        for metric in metrics:

            anova_values = anova[metric].to_numpy()
            doda_values = doda[metric].to_numpy()

            differences = doda_values - anova_values

            mean_difference = np.mean(differences)
            sd_difference = np.std(differences, ddof=1)

            if np.allclose(differences, 0):
                statistic = 0.0
                p_value = 1.0
            else:
                statistic, p_value = wilcoxon(
                    doda_values,
                    anova_values,
                    alternative="two-sided"
                )

            paired_d = (
                mean_difference / sd_difference
                if sd_difference > 0 else 0.0
            )

            comparison_rows.append({
                "Top_K": top_k,
                "Model": model_name,
                "Metric": metric,
                "ANOVA_Mean": np.mean(anova_values),
                "DODA_Mean": np.mean(doda_values),
                "Mean_Difference_DODA_minus_ANOVA": mean_difference,
                "Paired_SD": sd_difference,
                "Paired_Cohens_d": paired_d,
                "Wilcoxon_Statistic": statistic,
                "P_Value": p_value
            })

performance_comparison_df = pd.DataFrame(comparison_rows)

reject, adjusted_p, _, _ = multipletests(
    performance_comparison_df["P_Value"],
    alpha=0.05,
    method="holm"
)

performance_comparison_df["Holm_Adjusted_P"] = adjusted_p
performance_comparison_df["Significant"] = reject

display(performance_comparison_df.round(4))


In [ ]:
# =============================================================================
# STEP 34: SAVE STATISTICAL RESULTS
# =============================================================================

stats_dir = Path("../../../results/kidney_disease/statistics")
stats_dir.mkdir(parents=True, exist_ok=True)

feature_comparison_df.to_csv(
    stats_dir / "feature_set_comparison_5x5.csv",
    index=False
)

stability_df.to_csv(
    stats_dir / "feature_selection_stability_5x5.csv",
    index=False
)

performance_comparison_df.to_csv(
    stats_dir / "anova_vs_doda_predictive_statistics_5x5.csv",
    index=False
)

print(f"Saved statistical outputs to: {stats_dir.resolve()}")

feature_frequency_df.to_csv(
    stats_dir / "feature_selection_frequency_5x5.csv",
    index=False
)

kuncheva_df.to_csv(
    stats_dir / "kuncheva_pairwise_5x5.csv",
    index=False
)

kuncheva_summary_df.to_csv(
    stats_dir / "kuncheva_summary_5x5.csv",
    index=False
)

agreement_summary_df.to_csv(
    stats_dir / "anova_doda_agreement_summary_5x5.csv",
    index=False
)


In [ ]:
# =============================================================================
# STEP 36: FINAL SANITY CHECKS
# =============================================================================

print("=" * 70)
print("FINAL EXPERIMENT SANITY CHECK")
print("=" * 70)

print(f"Total predictors: {X.shape[1]}")
print(f"Top-K values: {TOP_K_VALUES}")
print(f"Missing values: {int(X.isna().sum().sum())}")
print(f"5-fold CV result rows: {len(cv5_results_df)}")
print(f"Repeated CV result rows: {len(cv_results_df)}")
print(f"Expected 5-fold rows: {5 * len(TOP_K_VALUES) * 2 * len(models)}")
print(f"Expected repeated CV rows: {25 * len(TOP_K_VALUES) * 2 * len(models)}")

assert X.shape[1] == 24
assert X.isna().sum().sum() == 0
assert TOP_K_VALUES == [5, 10, 15, 20]
assert len(cv5_results_df) == 5 * len(TOP_K_VALUES) * 2 * len(models)
assert len(cv_results_df) == 25 * len(TOP_K_VALUES) * 2 * len(models)
assert len(feature_frequency_df) == 2 * len(TOP_K_VALUES) * X.shape[1]
assert len(kuncheva_summary_df) == 2 * len(TOP_K_VALUES)
assert len(agreement_summary_df) == len(TOP_K_VALUES)

for top_k in TOP_K_VALUES:
    assert len(anova_results[top_k]["features"]) == top_k
    assert len(rank_doda_results[top_k]["features"]) == top_k

print("All sanity checks passed.")


# Methodological design notes for the BD-KDD experiment

### 1. Correct predictor count
BD-KDD has **25 processed columns**, but one is the binary target `Class`. Therefore the feature matrix contains **24 predictors**.

### 2. Top-K values
The experiment uses:

**Top-5, Top-10, Top-15, Top-20**

This provides approximately 20%, 40%, 60%, and 80% feature budgets over the 24 predictors. Top-24 is deliberately not used as a selection condition because retaining every feature cannot demonstrate feature-selection behavior.

### 3. Removed imputation
BD-KDD has zero missing values, so no imputation is performed.

### 4. Correct DODA architecture
ANOVA is configured with `k="all"` inside DODA. DODA receives all 24 statistical scores and performs the final Top-K selection.

### 5. No data leakage in CV
ANOVA and DODA are refit inside every training fold. Validation-fold data is never used to calculate feature-selection scores.

### 6. Model-specific scaling
Scaling is applied only to Logistic Regression and is fitted on the training data/fold. Random Forest and XGBoost use the selected features without scaling.

### 7. Consistent result paths
All outputs are stored under:

```text
results/
└── kidney_disease/
    ├── baseline/
    ├── doda/
    ├── cv/
    └── statistics/
```

### 8. Fixed split and repeated CV are separated
The fixed 80/20 experiment is an initial reproducible comparison. The 5×5 repeated stratified CV is the robustness analysis.

### 9. Feature-set change and rank change are separated
Jaccard measures feature-set overlap. DODA's rank-comparison output is retained separately because a feature can change rank without entering or leaving Top-K.

### 10. Clinical weights remain external
The clinical weights are loaded from the pre-specified JSON and are not derived from ANOVA scores, target correlations, model performance, or test-fold information.
